In [ ]:
#Install packages
system("conda install -y r-rlang=1.1.7 r-dplyr=1.1.4 r-ggplot2=3.5.1 r-tidyr=1.3.1 r-jsonlite=2.0.0 r-scales=1.4.0 r-viridis=0.6.5")

In [ ]:
#Load packages
library(dplyr)
library(ggplot2)
library(tidyr)
library(scales)
library(viridis)
library(jsonlite)

In [ ]:
#Load parameters
params <- fromJSON("galaxy_inputs/galaxy_inputs.json")
df_path <- as.character(params$table[1])

df <- read.table(df_path, header = TRUE, sep = "\t",
                  stringsAsFactors = FALSE, na.strings = c("NA", ""))

In [ ]:
###### General settings ######

CB_ORANGE <- "#E69F00"
CB_BLUE   <- "#56B4E9"
CB_DKBLUE <- "#0072B2"
CB_RED    <- "#D55E00"

EXCLUDE_POP <- c("mean", "All_populations_combined")


In [ ]:
########## Help functions ###########
# Warn (console message) when a species has no "mean" row, so the log
# makes it clear why the overlay is missing for that species -- the
# plot itself still gets produced.
warn_missing_mean <- function(df_subset, label) {
  all_species  <- unique(df_subset$Species_name)
  has_mean_sp  <- df_subset %>% filter(Pop == "mean") %>% pull(Species_name) %>% unique()
  missing      <- setdiff(all_species, has_mean_sp)
  if (length(missing) > 0) {
    message("NOTE [", label, "]: no 'mean' row for: ", paste(missing, collapse = ", "),
            " -> mean overlay omitted for these species (plot still produced).")
  }
}

# Pick facet_wrap(~Species) or facet_grid(Species ~ Marker_type) depending
# on whether more than one marker type is actually present in the data.
build_species_facet <- function(data, scales = "free") {
  if (dplyr::n_distinct(data$Marker_type) > 1) {
    facet_grid(Species_name ~ Marker_type, scales = scales)
  } else {
    facet_wrap(~ Species_name, scales = scales)
  }
}


In [ ]:
################# Ne graph #########################

df_clean    <- df %>% filter(!Pop %in% EXCLUDE_POP)

ne_points <- df_clean %>%
  transmute(
    Species     = Species_name,
    Marker_type = Marker_type,
    Ne          = log10(NeLD_final),
    CI_lo       = log10(JK_CI_down_final),
    CI_hi       = log10(JK_CI_up_final)
  ) %>%
  filter(!is.na(Ne))

warn_missing_mean(df %>% filter(!Pop %in% "All_populations_combined"), "Ne plot")

harmonic_mean <- function(x) {
  n <- length(x)
  x[is.na(x)] <- 999999 # NA signifie Infini
  return(n / sum(1 / x))
}

ne_summary <- df_clean %>%
  filter(Pop == "mean" | TRUE) %>%   # summary is computed from pop-level Ne regardless of a 'mean' row
  transmute(Species = Species_name, Ne = log10(NeLD_final),
            CI_lo = log10(JK_CI_down_final), CI_hi = log10(JK_CI_up_final)) %>%
  filter(!is.na(Ne)) %>%
  group_by(Species) %>%
  summarise(Ne = harmonic_mean(Ne, na.rm = TRUE), CI_lo = harmonic_mean(CI_lo, na.rm = TRUE),
            CI_hi = harmonic_mean(CI_hi, na.rm = TRUE), .groups = "drop")

ne_breaks     <- c(50, 500, 5000, 999999)
ne_labels     <- c("Ne=50", "Ne=500", "Ne=5000", "Ne=Inf")
ne_breaks_log <- log10(ne_breaks)

p1 <- ggplot() +
  theme_minimal() +
  geom_vline(xintercept = ne_breaks_log, linetype = "dashed", color = "grey50", linewidth = 0.4) +
  geom_point(
    data = ne_points,
    aes(y = Species, x = Ne, color = Species, shape = Marker_type),
    position = position_jitter(width = 0, height = 0.15), size = 2, alpha = 0.5
  ) +
  geom_pointrange(
    data = ne_summary,
    aes(y = Species, x = Ne, xmin = CI_lo, xmax = CI_hi, color = Species),
    size = 0.9, linewidth = 0.9, shape = 18
  ) +
  scale_color_viridis_d(option = "D", end = 0.9) +
  scale_x_continuous(breaks = ne_breaks_log, labels = ne_labels) +
  scale_y_discrete(limits = rev) +
  labs(
    title = "Effective population size (Ne) estimates",
    subtitle = "Mean +/- Jackknife 95% CI per species, points = Populations",
    x = "Ne (log10 scale)", y = "Species", color = "Species", shape = "Marker type"
  ) +
  theme_bw(base_size = 12) +
  theme(legend.position = "bottom", panel.grid.minor = element_blank())

In [ ]:
#################### Hexp/Hobs graphs ############################
df_mean <- df %>% filter(Pop == "mean")
warn_missing_mean(df %>% filter(!Pop %in% "All_populations_combined"), "Hobs/Hexp plot")

h_long <- df_mean %>%
  select(Species_name, Marker_type, Hobs, Hexp) %>%
  pivot_longer(cols = c(Hobs, Hexp), names_to = "Metric", values_to = "Value") %>%
  filter(!is.na(Value)) %>%
  mutate(Species_name = factor(Species_name, levels = sort(unique(Species_name))))

p2 <- ggplot(h_long, aes(x = Species_name, y = Value, fill = Metric)) +
  geom_col(position = "dodge", width = 0.7, colour = "white", linewidth = 0.3) +
  scale_fill_manual(values = c(Hobs = CB_ORANGE, Hexp = CB_BLUE)) +
  scale_y_continuous(expand = expansion(mult = c(0, 0.1))) +
  labs(title = "Observed and expected heterozygosity (mean)", x = "Species", y = "Heterozygosity", fill = NULL) +
  theme_bw(base_size = 12) +
  theme(axis.text.x = element_text(angle = 35, hjust = 1), legend.position = "top",
        panel.grid.major.x = element_blank(), panel.grid.minor = element_blank())

if (dplyr::n_distinct(h_long$Marker_type) > 1) {
  p2 <- p2 + facet_wrap(~ Marker_type)
}


In [ ]:
################ FIS (par population + moyenne) ##########################

fis_df <- df %>%
  filter(!Pop %in% "All_populations_combined") %>%
  select(Species_name, Marker_type, Pop, Fis) %>%
  filter(!is.na(Fis)) %>%
  mutate(
    Direction = ifelse(Fis >= 0, "Positive", "Negative"),
    IsMean    = ifelse(Pop == "mean", "Mean", "Population")
  ) %>%
  arrange(Species_name, Marker_type, IsMean, Fis) %>%
  mutate(Pop_id = factor(paste(Species_name, Marker_type, Pop, sep = "___"),
                         levels = paste(Species_name, Marker_type, Pop, sep = "___")))

warn_missing_mean(df %>% filter(!Pop %in% "All_populations_combined"), "Fis bar plot")

p3 <- ggplot(fis_df, aes(x = Fis, y = Pop_id, fill = Direction, alpha = IsMean)) +
  geom_col(width = 0.7, colour = "white", linewidth = 0.3) +
  geom_vline(xintercept = 0, linewidth = 0.8, colour = "grey20") +
  scale_fill_manual(values = c(Positive = CB_DKBLUE, Negative = CB_RED)) +
  scale_alpha_manual(values = c(Population = 0.45, Mean = 1),
                     guide = guide_legend(override.aes = list(fill = "grey40"))) +
  scale_y_discrete(labels = function(x) sub("^.*___", "", x)) +
  scale_x_continuous(labels = label_number(accuracy = 0.01), expand = expansion(mult = c(0.05, 0.05))) +
  labs(title = "Inbreeding coefficient (Fis) by population",
       subtitle = "Positive = excess homozygosity | Negative = excess heterozygosity",
       x = "Fis", y = NULL, fill = "Direction", alpha = "Bar type") +
  theme_bw(base_size = 12) +
  theme(panel.grid.major.y = element_blank(), panel.grid.minor = element_blank(),
        strip.text = element_text(face = "bold"))

p3 <- p3 + build_species_facet(fis_df, scales = "free_y")

n_species <- dplyr::n_distinct(fis_df$Species_name)

In [ ]:
################## Violin plot ##################
plot_single_metric_violin <- function(full_df, metric_col, title, y_lab = metric_col) {
  
  data_pop <- full_df %>%
    filter(!Pop %in% EXCLUDE_POP) %>%
    select(Species_name, Pop, Marker_type, all_of(metric_col)) %>%
    rename(Value = all_of(metric_col)) %>%
    filter(!is.na(Value))
  
  data_mean <- full_df %>%
    filter(Pop == "mean") %>%
    select(Species_name, Marker_type, all_of(metric_col)) %>%
    rename(MeanValue = all_of(metric_col)) %>%
    filter(!is.na(MeanValue))
  
  warn_missing_mean(full_df %>% filter(!Pop %in% "All_populations_combined"), title)
  
  multi_marker <- dplyr::n_distinct(data_pop$Marker_type) > 1
  
  p <- ggplot(data_pop, aes(x = Species_name, y = Value, fill = Species_name)) +
    geom_violin(alpha = 0.5, color = NA, scale = "width", trim = FALSE) +
    geom_jitter(aes(shape = Marker_type), width = 0.12, height = 0, size = 2.2,
                alpha = 0.8, color = "grey20") +
    scale_fill_viridis_d(option = "D", end = 0.85, guide = "none") +
    labs(title = title,
         subtitle = "Diamond = species-level mean (if available) | grey points = individual populations",
         x = NULL, y = y_lab, shape = "Marker type") +
    theme_minimal(base_size = 12) +
    theme(axis.text.x = element_text(angle = 30, hjust = 1),
          legend.position = "right", panel.grid.minor = element_blank())
  
  if (nrow(data_mean) > 0) {
    p <- p + geom_point(data = data_mean, aes(x = Species_name, y = MeanValue),
                        shape = 23, size = 4, fill = "white", color = "black",
                        stroke = 1, inherit.aes = FALSE)
  }
  
  if (multi_marker) {
    p <- p + facet_wrap(~ Marker_type)
  }
  
  p
}

p4 <- plot_single_metric_violin(df, "Fis", "Inbreeding coefficient (Fis) per species", "Fis")

p5 <- plot_single_metric_violin(df, "Ar", "Allelic richness (Ar) per species", "Ar")

In [ ]:
# ------------------------------------------------------------------
# 7. Pairwise differentiation indices (Fst / Gst / D Jost)
# ------------------------------------------------------------------
metric_labels <- c(average_pairwise_Fst = "Fst", average_Gst_Nei = "Gst (Nei)", average_Jost_D = "D (Jost)")
metric_order  <- c("Fst", "Gst (Nei)", "D (Jost)")

df_pops <- df %>%
  filter(!Pop %in% EXCLUDE_POP) %>%
  select(Species_name, Pop, Marker_type, all_of(names(metric_labels))) %>%
  pivot_longer(cols = all_of(names(metric_labels)), names_to = "Metric", values_to = "Value") %>%
  mutate(Metric = factor(recode(Metric, !!!metric_labels), levels = metric_order)) %>%
  filter(!is.na(Value))

df_species_mean <- df %>%
  filter(Pop == "mean") %>%
  select(Species_name, Marker_type, all_of(names(metric_labels))) %>%
  pivot_longer(cols = all_of(names(metric_labels)), names_to = "Metric", values_to = "MeanValue") %>%
  mutate(Metric = factor(recode(Metric, !!!metric_labels), levels = metric_order)) %>%
  filter(!is.na(MeanValue))

warn_missing_mean(df %>% filter(!Pop %in% "All_populations_combined"), "Pairwise indices plots")

pal <- viridisLite::viridis(length(metric_order), option = "D", end = 0.85)
names(pal) <- metric_order

# -- 7a. Bar chart -------------------------------------------------

df_pops_ordered <- df_pops %>%
  group_by(Species_name, Metric) %>%
  arrange(Value, .by_group = TRUE) %>%
  ungroup()

n_pops_per_species <- df %>% filter(!Pop %in% EXCLUDE_POP) %>% count(Species_name)
low_n <- n_pops_per_species %>% filter(n < 4)
if (nrow(low_n) > 0) {
  for (i in seq_len(nrow(low_n))) {
    message("WARNING: '", low_n$Species_name[i], "' has only ", low_n$n[i],
            " population(s) -> violin shape not statistically meaningful, raw points shown for transparency.")
  }
}

p6 <- ggplot(df_pops_ordered, aes(x = Metric, y = Value, fill = Metric)) +
  geom_violin(alpha = 0.5, color = NA, scale = "width", trim = FALSE) +
  geom_jitter(aes(shape = Marker_type), width = 0.08, height = 0, size = 2.2, alpha = 0.8, color = "grey20") +
  scale_fill_manual(values = pal, name = "Pairwise index") +
  labs(title = "Pairwise differentiation indices per population (violin plot)",
       subtitle = "Diamond = species-level mean (if available) | grey points = individual populations",
       x = NULL, y = "Value", shape = "Marker type") +
  theme_minimal(base_size = 12) +
  theme(strip.text = element_text(face = "bold"), legend.position = "right",
        panel.grid.minor = element_blank())

if (nrow(df_species_mean) > 0) {
  p6 <- p6 + geom_point(data = df_species_mean, aes(x = Metric, y = MeanValue),
                        shape = 23, size = 4, fill = "white", color = "black",
                        stroke = 1, inherit.aes = FALSE)
}

p6 <- p6 + facet_wrap(~ Species_name, scales = "free_y")

In [ ]:
##### Save plot #####

png(filename="outputs/Ne_forest_plot.png",
    width = 2400,
    height = 1800,
    res = 300)
print(p1)
dev.off()

png(filename="outputs/Hobs_Hex_barplot.png",
    width = 2400,
    height = 1800,
    res = 300)
print(p2)
dev.off()

png(filename="outputs/Fis_barplot.png",
    width = 2400,
    height = 1800,
    res = 300)
print(p3)
dev.off()

png(filename="outputs/Fis_violin.png",
    width = 2400,
    height = 1800,
    res = 300)
print(p4)
dev.off()

png(filename="outputs/Ar_violin.png",
    width = 2400,
    height = 1800,
    res = 300)
print(p5)
dev.off()

png(filename="outputs/pairwise_metrics_violin.png",
    width = 2400,
    height = 1800,
    res = 300)
print(p6)
dev.off()